In [1]:
import asyncio
import websockets
import json
import nest_asyncio
import time
import serial
import serial.tools.list_ports

ports = serial.tools.list_ports.comports()

for port, desc, hwid in sorted(ports):
        print("{}: {} [{}]".format(port, desc, hwid))




COM1: Communications Port (COM1) [ACPI\PNP0501\SIOBUAR2]
COM3: Intel(R) Active Management Technology - SOL (COM3) [PCI\VEN_8086&DEV_7AEB&SUBSYS_0A9F1028&REV_11\3&11583659&3&B3]
COM6: Arduino Mega 2560 (COM6) [USB VID:PID=2341:0042 SER=342393133353518022E0 LOCATION=1-1]
COM7: Arduino Mega 2560 (COM7) [USB VID:PID=2341:0042 SER=3423232363135150A1D2 LOCATION=1-5]


In [2]:
output_switch_1 = serial.Serial('COM6', 9600, write_timeout=0)  # open serial port
output_switch_2 = serial.Serial('COM7', 9600, write_timeout=0)  # open serial port


In [ ]:


nest_asyncio.apply()

last_state = {
    'cold_output_switch_1': None,
    'cold_output_switch_2': None,
    'cold_thru_switch_pair': None
}

commands = {
    'short': b's',
    'open': b'o',
    'load': b'l',
    'port1': b'1',
    'port2': b'2',
    'port3': b'3',
    'port4': b'4',
    'port5': b'5',
    'port6': b'6'
}

async def receive_data(websocket):
    async for message in websocket:
        try:
            instrument_data = json.loads(message)
            
            for key, val in instrument_data.items():
                if key in last_state and last_state[key] != val:
                    last_state[key] = val
                    print(f"Changed: {key} -> {val}")
                    
                    if key == 'cold_output_switch_1' and val in commands:
                        output_switch_1.write(commands[val])
                        
                    elif key == 'cold_output_switch_2' and val in commands:
                        output_switch_2.write(commands[val])
                            
        except Exception as e:
            print(f"Error: {e}")

async def main():
    async with websockets.serve(receive_data, "localhost", 8080):
        while True:
            await asyncio.sleep(0.1)

try:
    await main()
except KeyboardInterrupt:
    print("\nServer shutdown complete.")



Changed: cold_output_switch_1 -> short
Changed: cold_output_switch_2 -> open
Changed: cold_thru_switch_pair -> open
Changed: cold_output_switch_2 -> short
Changed: cold_output_switch_1 -> port1
Changed: cold_output_switch_2 -> port6
Changed: cold_output_switch_2 -> port4
Changed: cold_output_switch_1 -> short
